In [ ]:
!pip install earthengine-api


In [ ]:
import rasterio
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon
import requests
import xml.etree.ElementTree as ET
from io import BytesIO

import sys
sys.path.append('..')
import constants

import pystac
from pystac.extensions.table import TableExtension
from urllib.parse import urlparse, parse_qs
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension
from pystac.catalog import CatalogType
import datetime
import ee
ee.Initialize(project='ee-corestackdev')

In [ ]:
STAC_SAVE_DIR = "/home/vishnu/STAC-spec-raster/STAC-spec/data/PANindia_stac_raster"

SUB_COLLECTIONS = {} 

In [ ]:
def load_sheet_df(sheet_id: str, gid: str = "0"):
 
    csv_export_url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&gid={gid}'

    try:
        df_temp = pd.read_csv(csv_export_url)
        last_col_name = df_temp.columns[-1]

        df = pd.read_csv(csv_export_url, dtype={last_col_name: str}) 

        style_url_column = last_col_name

    except Exception as e:
        print(f"Error loading DataFrame: {e}")
        sys.exit(1)

    print("DataFrame Columns:", df.columns.tolist())
    print(f"Using column '{style_url_column}' for style file links.")
    
    return df, style_url_column


In [ ]:

sheet_id = "1rSg8Zm0RHQ7wgyVr7ZZoDj66CB88Om7gbXfqLhDK8HI"
df, STYLE_URL_COLUMN = load_sheet_df(sheet_id)


In [ ]:
def read_raster_data(image_info, ee_image):
    band_info = image_info['bands'][0]
    
    crs = band_info.get('crs')
    transform = band_info.get('crs_transform', [30, 0, 0, 0, -30, 0])
    bands = len(image_info['bands'])

    if "dimensions" in band_info:
        width = band_info['dimensions'][0]
        height = band_info['dimensions'][1]
    else:
        print(f"'dimensions' missing for asset. Estimating from geometry & scale")
        try:
            scale_x = abs(transform[0])
            scale_y = abs(transform[4])
            region = ee.Image(ee_image).geometry().bounds().getInfo()
            coords = region["coordinates"][0]
            min_x, min_y = coords[0]
            max_x, max_y = coords[2]
            width = int((max_x - min_x) / scale_x)
            height = int((max_y - min_y) / scale_y)
        except Exception as e:
            raise RuntimeError(f"Could not infer dimensions: {e}")

    min_x = transform[2]
    max_y = transform[5]
    max_x = min_x + width * transform[0]
    min_y = max_y + height * transform[4] 

    bbox = [min_x, min_y, max_x, max_y]
    footprint = Polygon([
        [min_x, min_y], [min_x, max_y],
        [max_x, max_y], [max_x, min_y],
        [min_x, min_y]
    ])
    
    try:
        gsd_meters = ee_image.projection().nominalScale().getInfo() 
    except Exception as e:
        print(f"Could not get nominal scale in meters for CRS {crs}. Fallback to crs_transform. Error: {e}")
        gsd_meters = abs(transform[0])

    shape = (bands, height, width)
    data_type = band_info['data_type']['type']

    return (bbox, mapping(footprint), crs, gsd_meters, shape, data_type)


In [ ]:
def create_raster_item(id, bbox, footprint, crs, shape): 
    
    raster_item = pystac.Item(id=id,
                              geometry=footprint,
                              bbox=bbox,
                              datetime=datetime.datetime.now(datetime.timezone.utc),
                              properties={})
    
    proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
    proj_ext.epsg = crs    
    proj_ext.shape = [shape[1], shape[2]]

    return (raster_item, None)

In [ ]:
def add_raster_data_asset(raster_item, url_string):
    raster_item.add_asset("data", Asset(
        href=url_string,
        roles=["data"],
        title="Asset Link"
    ))
    return raster_item

In [ ]:
def add_raster_extension(raster_item, gsd_val, data_type_val): 
    raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type_val,
        spatial_resolution=gsd_val,
    )
    raster_ext.bands = [raster_band]

In [ ]:
def parse_raster_style_file(style_file_url):
    
    try:
        response = requests.get(style_file_url)
        response.raise_for_status() 
    except requests.exceptions.RequestException as e:
        raise

    
    try:
        xml_string = response.content.decode('utf-8', errors='replace')
        style_content = BytesIO(xml_string.encode('utf-8'))
        tree = ET.parse(style_content)
    except ET.ParseError as e:
        try:
            print(f"Warning: Failed to parse XML with UTF-8 ({e}). Trying ISO-8859-1.")
            xml_string_alt = response.content.decode('iso-8859-1', errors='ignore')
            style_content = BytesIO(xml_string_alt.encode('utf-8'))
            tree = ET.parse(style_content)
        except ET.ParseError as e_final:
            raise e_final

    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry") + root.findall(".//item"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        if class_info:
            classes.append(class_info)
            
    if not classes:
         raise ValueError(f"No classification data (paletteEntry or item tags) found in style file.")
            
    return classes

In [ ]:
def add_classification_extension(raster_style_url, raster_item):
    
    style_info = parse_raster_style_file(style_file_url=raster_style_url)
    classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
    stac_classes = []
    
    for cls in style_info:
        color_hint = cls['color'].replace('#','') if cls.get('color') else None
        
        value = cls.get("value")
        if not isinstance(value, int):
             try:
                 value = int(float(str(value))) if value is not None else None
             except (TypeError, ValueError):
                 value = None
                 
        if value is None:
             print(f"Classification skipped for an entry with invalid value.")
             continue
             
        stac_class_obj = Classification.create(
            value=value,
            name=cls.get("label") or f"Class {value}",
            description=cls.get("label"),
            color_hint=color_hint
        )
        stac_classes.append(stac_class_obj)
        
    if stac_classes:
        classification_ext.classes = stac_classes
    else:
        raster_item.assets["data"].properties.pop("classification:classes", None)


    return (raster_item,style_info)

In [ ]:
def add_stylefile_asset(raster_item, style_file_url):
    raster_item.add_asset("style", Asset(
        href=style_file_url,
        media_type=MediaType.XML,
        roles=["metadata", "visualisation"],
        title="QGIS Style file"
    ))
    return raster_item

In [ ]:
def generate_raster_stac(image_info, url_string, style_file_url, layer_name, gsd, data_type, bbox, footprint, crs, shape): 
    
    stac_id = layer_name.replace(" ", "_").replace("/", "_").replace("-", "_").lower()
    
    
    raster_item, _ = create_raster_item(id=stac_id, bbox=bbox, footprint=footprint, crs=crs, shape=shape)
    
    raster_item = add_raster_data_asset(raster_item, url_string=url_string)
    
    add_raster_extension(raster_item, gsd_val=gsd, data_type_val=data_type)
    
    
    raster_item, style_info = add_classification_extension(raster_style_url=style_file_url,
                                                             raster_item=raster_item)
    
    raster_item = add_stylefile_asset(raster_item, style_file_url=style_file_url)
    
    return raster_item, style_info

In [ ]:
def create_root_and_collection():
   
    root_catalog = pystac.Catalog(
        id="PANindia",
        title="STAC Catalog for PAN India Assets",
        description="Root catalog for PAN India Assets and their metadata."
    )

   
    panindia_collection = pystac.Collection(
        id="panindia-layers",
        title="CoREstack Pan India Layers",
        description="A collection of various geospatial layers for Pan India.",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[68, 8, 98, 37]]), 
            temporal=pystac.TemporalExtent(
                [[constants.DEFAULT_START_DATE, constants.DEFAULT_END_DATE]]
            )
        ),
        license="https://spdx.org/licenses/CC-BY-4.0.html",
        providers=[
            pystac.Provider(
                name="CoREstack",
                roles=[
                    pystac.ProviderRole.PRODUCER,
                    pystac.ProviderRole.PROCESSOR,
                    pystac.ProviderRole.HOST,
                    pystac.ProviderRole.LICENSOR
                ],
                url="https://core-stack.org/"
            )
        ],
        keywords=["social-ecological", "sustainability", "CoRE stack"],
    )
    
    
    root_catalog.add_child(panindia_collection)

    return root_catalog, panindia_collection

In [ ]:
def create_sub_collection(title, parent_collection):
    
    collection_id = title.lower().replace(' ', '-').replace(':', '').replace('/', '-')
    
    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection (Theme): {title} ---")
        
        sub_collection = pystac.Collection(
            id=collection_id,
            title=title,
            description=f"{title}.",
            extent=parent_collection.extent,
            license=parent_collection.license,
            providers=parent_collection.providers
        )
        

        parent_collection.add_child(sub_collection)
        SUB_COLLECTIONS[collection_id] = sub_collection
        
    return SUB_COLLECTIONS[collection_id]

In [ ]:
def parse_asset_id(url_string, row_index):
    try:
        parsed_url = urlparse(url_string)
        query_params = parse_qs(parsed_url.query)
        return query_params['asset'][0]
    except Exception as e:
        print(f"Skipping row {row_index}: Could not parse asset ID from URL: {url_string}. Error: {e}")
        return None

In [ ]:
# def load_gee_image(asset_id):
#     try:
#         asset_metadata = ee.data.getAsset(asset_id)
#         asset_type = asset_metadata.get('type')

#         if asset_type == 'IMAGE':
#             print("Confirmed asset type: ee.Image.")
#             return ee.Image(asset_id)
#         elif asset_type == 'IMAGE_COLLECTION':
#             print("Confirmed asset type: ee.ImageCollection. Mosaicing for metadata.")
#             collection = ee.ImageCollection(asset_id)
#             return collection.mosaic()
#         else:
#             print(f"Asset type {asset_type} is not supported. Skipping.")
#             return None
#     except Exception as e:
#         print(f"Failed to load asset {asset_id}: {e}")
#         return None

In [ ]:
def load_gee_image(asset_id, mosaic_if_collection=True):
    """Load a GEE asset as ee.Image or mosaicked ee.Image with proper projection fix."""
    try:
        asset_metadata = ee.data.getAsset(asset_id)
        asset_type = asset_metadata.get('type')

        if asset_type == 'IMAGE':
            print("Confirmed asset type: ee.Image.")
            return ee.Image(asset_id)

        elif asset_type == 'IMAGE_COLLECTION':
            print("Confirmed asset type: ee.ImageCollection.")
            collection = ee.ImageCollection(asset_id)

            if mosaic_if_collection:
                # Pick one representative image for projection
                first_img = ee.Image(collection.first())
                proj = first_img.projection()
                scale = proj.nominalScale()
                crs = proj.crs()

                print(f"Using projection from first image: {crs}, scale: {scale.getInfo()} m")

                # Mosaic and reproject
                mosaicked = collection.mosaic().reproject(crs=crs, scale=scale)
                return mosaicked
            else:
                return collection

        elif asset_type == 'TABLE':
            print("Confirmed asset type: ee.FeatureCollection.")
            return ee.FeatureCollection(asset_id)

        else:
            print(f"Unsupported asset type: {asset_type}. Skipping.")
            return None

    except Exception as e:
        print(f"Failed to load asset {asset_id}: {e}")
        return None


In [ ]:
# img = load_gee_image("projects/corestack-datasets/assets/datasets/tree_health/modal_ch_2017")
# print(img.projection().nominalScale().getInfo())


In [ ]:
def process_layer(index, row, style_col_index, target_collection): 
    layer_name = row["Layer Name"]
    url_string = row["GEE asset link"]

    current_style_url = row.iloc[style_col_index]
    if not current_style_url or str(current_style_url).lower() in ('nan', 'none'):
        print(f"Skipping layer '{layer_name}'. Style file URL is missing.")
        return 0

    asset_id = parse_asset_id(url_string, index)
    if not asset_id:
        return 0

    print(f"Processing layer: {layer_name} (Asset ID: {asset_id})")

    try:
        image = load_gee_image(asset_id)
        if image is None:
            return 0

        image_info = image.getInfo()
        bbox, footprint, crs, gsd, shape, data_type = read_raster_data(image_info, ee_image=image)

        final_stac_item, style_data = generate_raster_stac(
            image_info=image_info,
            url_string=url_string,
            style_file_url=current_style_url,
            layer_name=layer_name,
            gsd=gsd, data_type=data_type, 
            bbox=bbox, footprint=footprint, crs=crs, shape=shape
        )

        target_collection.add_item(final_stac_item)
        print(f"Successfully generated STAC Item: {final_stac_item.id} (GSD: {gsd:.2f} m, Type: {data_type})")
        return 1

    except ee.EEException as e:
        print(f"GEE Error processing {layer_name} ({asset_id}): {e}. Skipping this item.")
    except Exception as e:
        print(f"General Error processing {layer_name}: {e}. Skipping this item.")
    return 0

In [ ]:
def save_catalog(root_catalog, generated_items_count):
    
    if generated_items_count > 0:
        print("Saving STAC Catalog")
        root_catalog.normalize_hrefs(STAC_SAVE_DIR)
        root_catalog.save(catalog_type=CatalogType.SELF_CONTAINED)
        print(f"SUCCESS: STAC Catalog saved to: {STAC_SAVE_DIR}")
        print(f"Total Items Generated: {generated_items_count}")
    else:
        print("No STAC Items were successfully generated.")

In [ ]:
def run_stac_generation(df, panindia_collection, root_catalog):
    
    print(f"Starting STAC generation for {len(df)} row")
    generated_items_count = 0
    
    style_col_index = df.columns.get_loc(STYLE_URL_COLUMN)
    
    current_collection = panindia_collection 

    for index, row in df.iterrows():
        layer_name = row["Layer Name"]
        gee_link = row["GEE asset link"]
        
    
        if pd.isna(gee_link):
            
            header_title = layer_name
            
            current_collection = create_sub_collection(
                title=header_title,
                parent_collection=panindia_collection 
            )
            print(f"Switching context to Collection: {header_title}")
            continue 
        
        else:
            
            if current_collection is panindia_collection:
                print(f"Layer '{layer_name}' encountered before explicit theme header. Adding to main panindia-layers collection.")

            target_collection = current_collection

            item_count = process_layer(
                index, 
                row, 
                style_col_index, 
                target_collection 
            )
            generated_items_count += item_count

    save_catalog(root_catalog, generated_items_count)



df, STYLE_URL_COLUMN = load_sheet_df(sheet_id) 

root_catalog, panindia_collection = create_root_and_collection()

run_stac_generation(df, panindia_collection, root_catalog)